In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer #for the age values
from sklearn.ensemble import RandomForestClassifier

data = pd.read_csv("Titanic-Dataset.csv")

# First, we start off by removing columns that we do not need
df = pd.DataFrame(data)
df.drop(columns=['PassengerId', 'Ticket', 'Name'], inplace=True) # Ticket column may have hidden information within it

In [ ]:
# Then we fill in empty values with nan/unknown values
df['Cabin'] = df['Cabin'].fillna('unknown')
# My response to Timo's suggestion to simply average the age
df['Age'] = df.groupby(['Pclass', 'Sex'])['Age'].transform(
    lambda x: x.fillna(x.median())
)
# Now the Age is made to be dependent on ticket class and sex - but only the missing values are filled, while the originals are left untouched

# Now we fill in the missing values for Embarked - we will learn a model to predict it based on Class and Fare

features = ['Pclass', 'Fare']
train = df[df['Embarked'].notna()]
test = df[df['Embarked'].isna()]

model = RandomForestClassifier()
model.fit(train[features], train['Embarked'])

df.loc[df['Embarked'].isna(), 'Embarked'] = model.predict(test[features])

In [ ]:
#Now we convert necessary columns into one-hot encodings
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})

# Now we one-hot encode the Embarked column
df = pd.get_dummies(df, columns=['Embarked'])

# Now, we need to handle Cabins - but some preparations are required
df['Deck'] = df['Cabin'].str.extract(r'([A-Za-z])', expand=False) # New column called Deck
df['Deck'] = df['Deck'].fillna('Unknown')
# Now we one-hot encode Deck
df = pd.get_dummies(df, columns=['Deck'], prefix='Deck')

In [ ]:
# Optional learing of Deck values - will need to remove the line that fills 'Unknown' in previous block if I decide to use thiss
features = ['Pclass', 'Fare', 'Age', 'Parch', 'SibSp']

train = df[df['Deck'].notna()]
test = df[df['Deck'].isna()]

model = RandomForestClassifier(random_state=42)
model.fit(train[features], train['Deck'])

In [ ]:
# Now we test out the "sacrificing yourself for loved ones theory"
#df.groupby('Parch')['Survived'].mean()
#df.groupby('SibSp')['Survived'].mean()

# Chat also highlighted the idea that being in a family would be beneficial and thus increase survival
df['Parch'].value_counts().sort_index()
df['SibSp'].value_counts().sort_index()

# Family size = number of siblings + number of spouses + number of parents + number of children + myself
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df.groupby('FamilySize')['Survived'].mean()

# It turns out that Chat's theory is correct

In [ ]:
# Investigation for embarked
# df.groupby(['Embarked', 'Pclass'])['Fare'].mean()
# df.groupby(['Embarked', 'Pclass']).size()

df['FareBin'] = pd.qcut(df['Fare'], q=4)

embarked_map = (
    df.dropna(subset=['Embarked'])
      .groupby(['Pclass', 'FareBin'])['Embarked']
      .agg(lambda x: x.mode()[0])
)

def fill_embarked(row):
    if pd.isna(row['Embarked']):
        return embarked_map.get((row['Pclass'], row['FareBin']), 'S')  # fallback
    return row['Embarked']

df['Embarked'] = df.apply(fill_embarked, axis=1)

# The above was abandoned to instead build a model to predict the missing Embarked values